# SentinelRUL: Multitask Training

Fine tunes the shared GRU backbone jointly on RUL regression and sensor forecasting, starting from the
forecast-pretrained weights produced by `02_kaggle_forecast_training.ipynb`. Run this on Kaggle with GPU
accelerator. Download `multitask_best.pt` from `/kaggle/working/` when finished.

In [ ]:
!pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import os
import math
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print(f'torch {torch.__version__}  cuda: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using device: {DEVICE}')

## Data and Checkpoint Configuration

Point `DATA_DIR` at the Kaggle dataset input folder for the NASA C-MAPSS data, and `FORECAST_CKPT`
at the `forecast_best.pt` file from Stage 6. Upload it via *Add Data > Upload* as a private dataset
and update the path below to match.

In [ ]:
DATA_DIR       = '/kaggle/input/nasa-cmapss-data'
FORECAST_CKPT  = '/kaggle/input/sentinelrul-forecast-checkpoint/forecast_best.pt'

ALL_COLS = ['engine_id', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

DROP_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
SENSOR_COLS  = [c for c in ALL_COLS if c.startswith('s') and c not in DROP_SENSORS]

WINDOW_SIZE      = 30
FORECAST_HORIZON = 5
RUL_CLIP         = 125
VAL_ENGINES      = 20
SEED             = 42
ALPHA            = 0.5

print(f'{len(SENSOR_COLS)} sensors kept: {SENSOR_COLS}')